# Module 5 Songs Cleaning Worked Examples

A separate, fully worked companion to **handout sections 5.4–5.7**. Each
short explanation is followed by a code cell and its saved output. Predict
the change, run the cell, and compare the result with the original values.

This notebook uses **twelve fictional songs** and invented messy metadata.
It covers text cleaning, numbers, dates, missing values, and a supplied regex.
It does **not** solve the Taylor Swift, Olivia Dean, and BTS homework.

## Open and run

Unzip the supplied folder and keep this notebook beside its `data` folder.
Open the folder in VS Code, open the notebook, and select your existing
Module 5 Python environment with pandas 2.0 or newer. Use **Restart Kernel**,
then **Run All**. No API, internet connection, downloaded lyrics, or loop is
needed. Alternatively, copy the notebook beside the `data` folder in your
existing `05-wrangle` checkout. Do not submit this worked notebook on Gradescope.

The first code cell imports pandas. The next reads `data/songs_messy.csv`.
`messy_raw` remains unchanged; `clean` is the working copy.

In [1]:
import pandas as pd
from IPython.display import display

#### 5.4 Load the messy export

Keep every column as a string for inspection. `keep_default_na=False` preserves
blank strings and `"N/A"` instead of automatically treating them as missing.
One row represents one song; `song_id` is its unique identifier.

In [2]:
messy_raw = pd.read_csv(
    "data/songs_messy.csv",
    dtype="string", keep_default_na=False
)
clean = messy_raw.copy()
display(messy_raw.head())
print("Rows:", len(messy_raw))

,song_id,song,artist,play_count,added_on,track_code,genre,playlist_note
0,D01,Morning Train,Paper Lanterns,"1,250 plays",2026-09-03,POP-STUDIO-001,pop,
1,D02,Window Light,PAPER LANTERNS,800 plays,"Sep 4, 2026",POP-LIVE-002,pop,Replay D02 tonight
2,D03,Blue Bicycle,Harbor Lines,N/A,2026/09/05,FOLK-STUDIO-003,folk,No song mentioned
3,D04,After the Rain,harbor lines,300 plays,2026-09-06,FOLK-LIVE-004,,Old D01; replacement D04
4,D05,Late Library,NIGHT ATLAS,unknown,not recorded,ROCK-STUDIO-005,rock,Please check d05


Rows: 12


#### Remove surrounding spaces with .str.strip()

`strip()` removes whitespace from the beginning and end of each string. It
does not remove spaces between words. The brackets in the comparison table
make surrounding spaces easier to see. Predict what changes before running.

In [3]:
clean["artist"] = clean["artist"].str.strip()
display(pd.DataFrame({
    "before_strip": "[" + messy_raw["artist"] + "]",
    "after_strip": "[" + clean["artist"] + "]"
}).head())

,before_strip,after_strip
0,[ Paper Lanterns ],[Paper Lanterns]
1,[PAPER LANTERNS],[PAPER LANTERNS]
2,[ Harbor Lines],[Harbor Lines]
3,[harbor lines ],[harbor lines]
4,[NIGHT ATLAS ],[NIGHT ATLAS]


#### Standardize capitalization with .str.lower()

`lower()` converts letters to lowercase. It lets `PAPER LANTERNS` and
`Paper Lanterns` become the same grouping value. Assign the result back to
the column; calling the method alone would not update the stored values.

In [4]:
clean["artist"] = clean["artist"].str.lower()
display(pd.DataFrame({
    "original_artist": messy_raw["artist"],
    "cleaned_artist": clean["artist"]
}).head())

,original_artist,cleaned_artist
0,Paper Lanterns,paper lanterns
1,PAPER LANTERNS,paper lanterns
2,Harbor Lines,harbor lines
3,harbor lines,harbor lines
4,NIGHT ATLAS,night atlas


#### Split a structured track code

`"POP-STUDIO-001"` splits into three strings. `.str[0]` selects the first item;
`.str[-1]` selects the last. Keep `"001"` as a string to retain its leading zeros.
Predict the two new columns before running the cell.

In [5]:
clean["genre_code"] = clean["track_code"].str.split("-").str[0]
clean["item_code"] = clean["track_code"].str.split("-").str[-1]
display(clean[["track_code", "genre_code", "item_code"]].head())

,track_code,genre_code,item_code
0,POP-STUDIO-001,POP,001
1,POP-LIVE-002,POP,002
2,FOLK-STUDIO-003,FOLK,003
3,FOLK-LIVE-004,FOLK,004
4,ROCK-STUDIO-005,ROCK,005


#### Find literal text

`contains` checks each track code. `regex=False` means literal text, and
`na=False` treats missing text as a non-match. Which rows will the mask keep?

In [6]:
pop_mask = clean["track_code"].str.contains("POP-", regex=False, na=False)
display(clean.loc[pop_mask, ["song_id", "track_code"]])

,song_id,track_code
0,D01,POP-STUDIO-001
1,D02,POP-LIVE-002
6,D07,POP-STUDIO-007
7,D08,POP-LIVE-008


#### 5.5 Remove formatting from play counts

These are invented counts for one shared reporting period, not API results.
First remove the unit and comma: `"1,250 plays"` should become `"1250"`.
At this step the values are still strings.

In [7]:
plays_text = (
    clean["play_count"].str.strip()
    .str.replace(" plays", "", regex=False)
    .str.replace(",", "", regex=False)
)
display(plays_text.head())

0       1250
1        800
2        N/A
3        300
4    unknown
Name: play_count, dtype: string

#### Convert the cleaned text to numbers

`pd.to_numeric` performs the conversion. `errors="coerce"` makes values it
cannot parse missing; it does not make them zero. Compare `D11`, whose count
is genuinely zero, with rows whose counts are unknown.

In [8]:
clean["play_count"] = pd.to_numeric(plays_text, errors="coerce")
display(pd.DataFrame({
    "original_count": messy_raw["play_count"],
    "numeric_count": clean["play_count"]
}))

,original_count,numeric_count
0,"1,250 plays",1250.0
1,800 plays,800.0
2,N/A,<NA>
3,300 plays,300.0
4,unknown,<NA>
5,600 plays,600.0
6,"1,000 plays",1000.0
7,850 plays,850.0
8,450 plays,450.0
9,,<NA>


#### Convert mixed dates and inspect invalid entries

`added_on` is a playlist date, not a release date. `format="mixed"` parses
different formats; `errors="coerce"` makes invalid entries `NaT`.
Successful parsing cannot resolve an ambiguous date such as `03/04/2026`:
check whether the source means March 4 or April 3.

In [9]:
clean["added_on"] = pd.to_datetime(
    clean["added_on"], format="mixed", errors="coerce"
)
display(pd.DataFrame({
    "original_date": messy_raw["added_on"],
    "converted_date": clean["added_on"]
}))

,original_date,converted_date
0,2026-09-03,2026-09-03
1,"Sep 4, 2026",2026-09-04
2,2026/09/05,2026-09-05
3,2026-09-06,2026-09-06
4,not recorded,NaT
5,"September 8, 2026",2026-09-08
6,2026-09-09,2026-09-09
7,"Sep 10, 2026",2026-09-10
8,2026/09/11,2026-09-11
9,,NaT


#### Use the datetime accessor

`.dt` provides operations on datetime values. `strftime("%Y-%m")` creates a
year-month string. A missing date remains missing.

In [10]:
clean["month"] = clean["added_on"].dt.strftime("%Y-%m")
display(clean[["song_id", "added_on", "month"]].head())

,song_id,added_on,month
0,D01,2026-09-03,2026-09
1,D02,2026-09-04,2026-09
2,D03,2026-09-05,2026-09
3,D04,2026-09-06,2026-09
4,D05,NaT,NaN


#### 5.6 Locate missing values

`isna()` identifies recognized missing values; summing counts them.
Use the resulting mask to inspect the **original** text that failed conversion.
`notna()` identifies present values. Blank strings are not themselves missing
markers in the raw table we loaded.

In [11]:
display(clean[["play_count", "added_on"]].isna().sum())
display(messy_raw.loc[
    clean["play_count"].isna(), ["song_id", "play_count"]
])
print("Known play counts:", clean["play_count"].notna().sum())

play_count    3
added_on      2
dtype: int64

,song_id,play_count
2,D03,N/A
4,D05,unknown
9,D10,


Known play counts: 9


#### Drop rows only when a required value is missing

For a calculation that needs a play count, create a separate table with
`dropna(subset=["play_count"])`. This does not drop rows from `clean` or require
every other column to be complete. A total of recorded plays excludes unknown
counts; it is not necessarily the true total for all twelve songs.

In [12]:
known_plays = clean.dropna(subset=["play_count"])
print("All songs:", len(clean))
print("Songs with a known count:", len(known_plays))
print("Recorded plays:", known_plays["play_count"].sum())

All songs: 12
Songs with a known count: 9
Recorded plays: 6150.0


#### Convert known placeholders to missing values

In this source, `"N/A"` and `""` in `genre` mean unknown. `replace` operates on
the whole value here, unlike `.str.replace`, which replaces text within a value.

In [13]:
clean["genre"] = clean["genre"].replace(["N/A", ""], pd.NA)
display(clean.loc[clean["genre"].isna(), ["song_id", "genre"]])

,song_id,genre
3,D04,<NA>
7,D08,<NA>


#### Keep unknown genres visible

`fillna("Unknown")` gives missing genres an explicit category. This is a
reporting choice, not a claim that we discovered each song's true genre.

In [14]:
clean["genre"] = clean["genre"].fillna("Unknown")
display(clean[["song_id", "genre"]])

,song_id,genre
0,D01,pop
1,D02,pop
2,D03,folk
3,D04,Unknown
4,D05,rock
5,D06,rock
6,D07,pop
7,D08,Unknown
8,D09,folk
9,D10,folk


#### Match a complete favorites list

Suppose the following is one listener's **complete** saved-song list at a
reporting cutoff. A left merge preserves every song. A missing match means
not saved **only because this list is complete**. It is not evidence about
missing lyrics or unknown play counts.

Run the demonstration from its setup cell again if you need to repeat the
merge; otherwise `is_saved` already exists and the merge would add suffixes.

In [15]:
favorites = pd.DataFrame({
    "song_id": ["D01", "D03", "D05"],
    "is_saved": [True, True, True]
})
clean = clean.merge(
    favorites, on="song_id", how="left", validate="one_to_one"
)
display(clean[["song_id", "is_saved"]])

,song_id,is_saved
0,D01,True
1,D02,NaN
2,D03,True
3,D04,NaN
4,D05,True
5,D06,NaN
6,D07,NaN
7,D08,NaN
8,D09,NaN
9,D10,NaN


#### Fill a justified Boolean absence

Use nullable Boolean type, then fill the unmatched favorites with `False`.
Do not apply this rule to unknown play counts or unavailable lyrics.

In [16]:
clean["is_saved"] = clean["is_saved"].astype("boolean").fillna(False)
display(clean[["song_id", "is_saved"]])

,song_id,is_saved
0,D01,True
1,D02,False
2,D03,True
3,D04,False
4,D05,True
5,D06,False
6,D07,False
7,D08,False
8,D09,False
9,D10,False


#### Interpret a Boolean mean

The mean of Boolean values is the fraction that are `True`. Here `saved_share`
uses **songs in this playlist** as its denominator, not plays or all songs in
an artist's catalog. Recorded plays still omit missing counts.

In [17]:
artist_report = clean.groupby("artist", as_index=False).agg(
    songs=("song_id", "size"),
    recorded_plays=("play_count", "sum"),
    saved_share=("is_saved", "mean")
)
display(artist_report)

,artist,songs,recorded_plays,saved_share
0,harbor lines,4,750.0,0.25
1,night atlas,4,1500.0,0.25
2,paper lanterns,4,3900.0,0.25


#### 5.7 Recognize a regex in playlist notes

Regex describes a text pattern. You are not expected to memorize or write
patterns. This supplied pattern matches uppercase `D` followed by digits,
with word boundaries. Parentheses capture the ID; `extract` selects the first
match. With one capture group, `expand=False` returns a Series.

Predict what happens with lowercase `d05`, a blank note, and two IDs in one
note. The `r` prefix preserves the backslashes in the Python string.

In [18]:
notes = messy_raw["playlist_note"].head()
song_ids = notes.str.extract(r"\b(D[0-9]+)\b", expand=False)
display(pd.DataFrame({"playlist_note": notes, "extracted_id": song_ids}))

,playlist_note,extracted_id
0,,<NA>
1,Replay D02 tonight,D02
2,No song mentioned,<NA>
3,Old D01; replacement D04,D01
4,Please check d05,<NA>


#### Explain what the extraction means

The first ID in `"Old D01; replacement D04"` is not the replacement ID. A match
also does not prove an ID exists in the songs table. Ask AI to explain the
pattern, then check its explanation against the displayed rows.

For practice, explain why `size` and `count` could differ in an artist report,
and why recorded plays may not establish which artist has the most actual
plays. These recorded plays are invented practice data, not the real-song homework.

## Check the final result

The cleaning preserves all **12 songs**. There are **3 missing play counts**
and **2 missing dates**. The 9 known counts sum to **6,150 recorded plays**;
this is not a complete total because the other counts are unknown. `D11`
has a real zero count and must not be confused with a missing value.

The table below keeps the missing values visible. The CSV on disk has not
been changed. To repeat the full pipeline, restart the kernel and run all
cells in order.

In [19]:
display(clean[[
    "song_id", "artist", "play_count", "added_on", "genre", "is_saved"
]])

,song_id,artist,play_count,added_on,genre,is_saved
0,D01,paper lanterns,1250.0,2026-09-03,pop,True
1,D02,paper lanterns,800.0,2026-09-04,pop,False
2,D03,harbor lines,<NA>,2026-09-05,folk,True
3,D04,harbor lines,300.0,2026-09-06,Unknown,False
4,D05,night atlas,<NA>,NaT,rock,True
5,D06,night atlas,600.0,2026-09-08,rock,False
6,D07,paper lanterns,1000.0,2026-09-09,pop,False
7,D08,paper lanterns,850.0,2026-09-10,Unknown,False
8,D09,harbor lines,450.0,2026-09-11,folk,False
9,D10,harbor lines,<NA>,NaT,folk,False
